## GENeSYS-MOD RES Tool - APAC hub (batch/tiling workflow)

Continent-scale run at country level over 18 APAC countries (Australia, Bangladesh,
Brunei, Indonesia, Japan, Cambodia, South Korea, Laos, Myanmar, Malaysia, New Zealand,
Papua New Guinea, Philippines, North Korea, Singapore, Thailand, Timor-Leste,
Vietnam). The hub spans ~88-179 E x -55-46 N - bigger than the Europe hub - so a single
cutout over the whole extent (plus rasterizing land cover over it) would exhaust a
laptop's RAM. This driver **tiles** the area into disjoint country groups, builds one
small ERA5 cutout per tile, runs the full pipeline (capacity factors + GIS potentials)
on that tile, frees the cutout, then **combines everything into single outputs** (one
CSV per tech/category, one combined potentials table, one combined map). Peak RAM is
bounded by the largest single tile (`australia`, ~41k cells at 0.275 deg), not the
whole extent.

All the batch logic is `run_batch_by_tile` in `functions.py`; this notebook only sets
paths/parameters and the editable `TILES` dict. Region-geojson workflow (a literal
`region` column, no `iso_a2`).

For a quick single-country proof of concept instead of the full batch, see
`GENeSYS-MOD_RES_Tool_SG.ipynb` (Singapore, explicit cutout corners, no tiling needed).

Data layers:
- **regions**: `geodata/regions/fel_APAC_regions.geojson` (18 hub countries, `region` =
  ISO-A2-ish code, e.g. AU/ID/SG/...).
- **land cover**: ESA WorldCover 2021 v200, built PER TILE (`build_worldcover_apac.py`,
  one 10 m source-tile fetch + downsample + merge per country group - the full hub
  would mean ~1000+ candidate 10 m tiles for one raster). Decimation scales with tile
  size: ~100 m for small tiles, ~300 m for the continent-scale ones (australia,
  east_asia, maritime_sea).
- **protected areas**: one merged, strict-tier (IUCN Ia/Ib/II) WDPA GeoPackage for the
  WHOLE hub (`build_wdpa_apac.py` -> `wdpa_apac.gpkg`). Protected Planet's cloudfront
  bucket no longer reliably publishes a plain per-country pack for every country (only
  3 of 18 APAC countries have one at the current release), so this is built from the
  ~2 GB global WDPA shapefile pack, discovered live (its hash suffix changes every
  release) and filtered to the hub bbox + strict tier.
- **EEZ**: one merged, hub-wide EEZ polygon layer (`build_eez_apac.py` ->
  `eez_apac.geojson`, clipped to the hub bbox and simplified - Indonesia's archipelagic
  EEZ is ~368k vertices unsimplified, which GDAL's GeoJSON reader refuses outright).
  `_get_coords_custom` clips it further to each tile's own cutout bbox.
- **bathymetry**: GMRT GridServer pull PER COASTAL TILE (`build_bathymetry_apac.py` ->
  `bathymetry_apac_{tile}.nc`). Every APAC tile is coastal (Laos is the hub's only
  landlocked country, grouped with four coastal neighbours in `mainland_sea`).

## General requirements & preparation

This timeseries script uses functions from atlite to generate weather timeseries for GENeSYS-MOD. It uses the ERA5 weather dataset, which is downloaded and transformed locally.
Please be aware that for a continent-scale hub like APAC, a large amount of system memory is required (at least 24GB of RAM) for the biggest single tile.

To use this script, you need to have the following packages installed:
- numpy
- matplotlib
- seaborn
- pandas / geopandas
- scikit-learn
- cartopy
- xarray
- atlite

In addition, in order to be able to load the ERA5 cutouts, you need an API key from https://cds.climate.copernicus.eu/user/register?destination=%2F%23!%2Fhome

To activate the API key, follow the instructions at https://cds.climate.copernicus.eu/api-how-to

For more information on how to run atlite, please refer to https://atlite.readthedocs.io/en/latest/

In [ ]:
from functions import *

In [ ]:
onshore_turbine = 'Vestas_V112_3MW'
offshore_turbine = 'Vestas_V164_7MW_offshore'
solar_panel = 'CSi'

# APAC hub geometry (18 countries, region = ISO-A2-ish code). The 'region' column
# (and NO 'iso_a2') triggers the region-geojson workflow the batch process needs.
geo_file = "geodata/regions/fel_APAC_regions.geojson"

# Offshore. One whole-hub EEZ polygon (built by build_eez_apac.py from eez_v12.gpkg
# by ISO_SOV1, clipped to the hub bbox and simplified). _get_coords_custom clips it to
# each tile's cutout bbox, so one file serves every tile. Bathymetry is PER TILE
# (geodata/offshore/bathymetry_apac_{tile}.nc, built by build_bathymetry_apac.py) and wired
# through the TILE_LAYERS dict below, so the global bathymetry_file stays None.
offshore_file = "geodata/offshore/eez_apac.geojson"
bathymetry_file = None

# 'Prove fast, scale later': one day first to smoke-test the whole tiling machinery;
# switch to "2018" for a full year (cutouts + ERA5 chunk cache are reused).
timeframe = "2018-01-01"
filename = "apac"            # BASE prefix; per-tile calls use f"{filename}_{tile}"
output_dir = create_output_folder(timeframe)

admin = 0                    # country level (the only level provided in the geojson)

# Availability-map colour: a hex ('#004664' -> white->navy ramp) or a named
# matplotlib colormap ('Greens'). None = default Greens.
map_color = '#004664'

In [ ]:
# Cutout resolution (degrees). Coarse (~0.275 deg ~= 30 km); ERA5 is ~0.25-0.3 deg
# native anyway, so this is a sensible continent-scale default. Singapore and other
# small island tiles still get at least a few cells at this resolution once grouped
# with their larger neighbours in a tile.
dx_step = 0.275
dy_step = 0.275

# Buffer (degrees) added around each tile's country polygons when sizing its cutout,
# so coastal cells / border cells are not clipped. Overlaps between neighbouring
# tiles' buffers are fine - the shared ERA5 chunk cache reuses the downloads.
bbox_buffer = 1.0

# PV slope/azimuth for utility-scale (ground-mounted) PV.
pv_slope = 20              # APAC hub spans both hemispheres widely - a single fixed
pv_azimuth = 180           # slope/azimuth is a compromise; pv_optimal_tilt overrides it.
pv_optimal_tilt = True
# Rooftop PV, split like the model's P_PV_Rooftop_Residential / _Commercial. Each
# segment gets its own CF timeseries from a mix of panel orientations (tilt in
# degrees or "optimal"; azimuth "equator" = facing the equator per cell, "east",
# "west", or degrees from north) and a share of the rooftop potential.
# Residential roofs face wherever the house does; commercial roofs are mostly flat,
# so racks can face the equator at the optimal tilt.
rooftop_segments = {
    "residential": [{"tilt": 20, "azimuth": "equator", "weight": 0.5},
                    {"tilt": 20, "azimuth": "east", "weight": 0.25},
                    {"tilt": 20, "azimuth": "west", "weight": 0.25}],
    "commercial": [{"tilt": "optimal", "azimuth": "equator", "weight": 1.0}],
}
# Share of the rooftop potential per segment, per country, from GHSL built-up
# surface (geodata/config/rooftop_shares.csv, built by build_rooftop_shares.py).
rooftop_shares = load_rooftop_shares(geo_file=geo_file)   # region codes are ISO2

# Usable-site thresholds (min developable share of a cell to count as a usable site).
usable_threshold = 0.01
usable_threshold_rooftop = 0.05

## TILES - the one thing you edit

Group the countries into tiles (country groups). Each tile becomes one small cutout.
**Every region must appear in exactly one tile** (disjoint - `run_batch_by_tile`
asserts this to prevent double-counting). Keep tiles geographically compact so each
cutout stays small; a tile with more, spread-out countries makes a bigger cutout.

The default below covers all 18 regions in `fel_APAC_regions.geojson`. New Zealand and
Papua New Guinea get their own single-country tiles despite being small - they sit
~5000 km apart (and from the other tiles) with open ocean between, so merging them
with a neighbour would waste cutout cells on empty sea. Australia is its own tile -
continent-scale on its own (~41k cells at 0.275 deg, already the largest single tile,
comparable to the Europe hub's biggest).

In [ ]:
TILES = {
    "mainland_sea": ["TH", "LA", "KH", "VN", "MM"],   # Thailand/Laos/Cambodia/Vietnam/Myanmar
    "maritime_sea": ["MY", "SG", "BN", "ID", "TL"],   # Malaysia/Singapore/Brunei/Indonesia/Timor-Leste
    "philippines":  ["PH"],
    "east_asia":    ["JP", "KR", "KP"],               # Japan/South Korea/North Korea
    "south_asia":   ["BD"],                           # Bangladesh
    "australia":    ["AU"],
    "new_zealand":  ["NZ"],
    "png":          ["PG"],                           # Papua New Guinea
}

## Exclusion layers (land cover + protected areas)

Region-agnostic builders; only the paths/codes are region-specific. Each tile names
its own ESA WorldCover raster (`build_worldcover_apac.py <tile>`, run once per tile
before using it here) through the `TILE_LAYERS` dict below; `functions.py` builds that
tile's excluders lazily and frees them with the cutout. No region logic lives in
`functions.py` - only these generic per-tile specs.

Protected areas use ONE merged, strict-tier WDPA GeoPackage for the whole hub
(`build_wdpa_apac.py`), shared across every tile (the vector-read cache in
`functions.py` reads+filters it once).

In [ ]:
# === Land-cover + protected-area layers for GIS potentials (per-tile) ===
# ESA WorldCover (EPSG:4326, degrees) for every tile - its excluder MUST use a
# projected CRS, so raster_crs="__equal_area__" (-> functions.py uses
# equal_area_crs(tile_shapes)) is mandatory here, else land availability comes back
# ZERO everywhere (CLAUDE.md projected-CRS gotcha). Classes: 10 tree, 20 shrub,
# 30 grass, 40 crop, 50 built, 60 bare, 70 snow, 80 water, 90 wetland, 95 mangrove,
# 100 moss, 0 ocean/nodata fill. Strict-exclude 0/tree/crop/built/water/wetland/
# mangrove; keep shrub/grass/bare developable (drop crop(40) from the list if you
# want to allow agrivoltaics). Rooftop proxy keeps built-up 50.
WC_EXCLUDE = [0, 10, 40, 50, 80, 90, 95]
WC_ROOFTOP = [50]

def worldcover_tile_raster(tile):
    # build_worldcover_apac.py tags each tile with its own resolution (100-300 m), so
    # look the file up rather than assume one.
    import glob
    found = glob.glob(f"geodata/landcover/worldcover_apac_{tile}_*m.tif")
    assert len(found) == 1, f"expected one WorldCover raster for tile {tile}, found {found}"
    return found[0]

# Protected areas - one merged, strict-tier WDPA GeoPackage for the whole hub (built
# by build_wdpa_apac.py from the WDPA global pack, filtered to IUCN Ia/Ib/II at build
# time). Terrestrial+coastal strict areas exclude land; marine+coastal strict areas
# exclude offshore.
WDPA = "geodata/exclusions/wdpa_apac.gpkg"
WDPA_LAYER = "wdpa"
PROTECTED_QUERY = "REALM in ['Terrestrial', 'Coastal']"      # WDPA already IUCN Ia/Ib/II
OFFSHORE_QUERY  = "REALM in ['Marine', 'Coastal']"            # (build-time IUCN filter)


def worldcover_tile(tile, bathymetry=None, offshore_file="__default__"):
    """A TILE_LAYERS spec for a WorldCover-covered tile. bathymetry -> per-tile depth
    grid; offshore_file left at the sentinel uses the global EEZ (pass None to force
    landlocked)."""
    spec = {
        "land_raster": worldcover_tile_raster(tile),
        "land_exclude_codes": WC_EXCLUDE,
        "rooftop_codes": WC_ROOFTOP,
        "raster_crs": "__equal_area__",           # geographic raster -> per-tile LAEA
        "protected_files": WDPA,
        "protected_layer": WDPA_LAYER,
        "protected_query": PROTECTED_QUERY,
        "offshore_exclude_files": WDPA,
        "offshore_exclude_layer": WDPA_LAYER,
        "offshore_exclude_query": OFFSHORE_QUERY,
    }
    if bathymetry is not None:
        spec["bathymetry_file"] = bathymetry
    if offshore_file != "__default__":
        spec["offshore_file"] = offshore_file
    return spec


def bathy(tile):
    return f"geodata/offshore/bathymetry_apac_{tile}.nc"


# Per-tile layer map. Every APAC tile is coastal (see module docstring), so every tile
# gets its own GMRT bathymetry file. Keys must match TILES.
TILE_LAYERS = {
    "mainland_sea": worldcover_tile("mainland_sea", bathymetry=bathy("mainland_sea")),
    "maritime_sea": worldcover_tile("maritime_sea", bathymetry=bathy("maritime_sea")),
    "philippines":  worldcover_tile("philippines",  bathymetry=bathy("philippines")),
    "east_asia":    worldcover_tile("east_asia",    bathymetry=bathy("east_asia")),
    "south_asia":   worldcover_tile("south_asia",   bathymetry=bathy("south_asia")),
    "australia":    worldcover_tile("australia",    bathymetry=bathy("australia")),
    "new_zealand":  worldcover_tile("new_zealand",  bathymetry=bathy("new_zealand")),
    "png":          worldcover_tile("png",          bathymetry=bathy("png")),
}

# Global excluders stay None: every tile supplies its own via TILE_LAYERS, so the
# global ones are never used. run_potentials on -> GIS potentials run per tile.
excluder = cities = wind_excluder = None
run_potentials = True
print(f"run_potentials = {run_potentials}; {len(TILE_LAYERS)} tiles have layer specs")
print("  tiles:", list(TILE_LAYERS.keys()))

## Capacity densities (MW/km2) and developable share per tech

Placeholder figures (DEA 2030 utility-scale / generic) shared across all tiles -
swap for region-specific assumptions once you have better sources per country.

In [ ]:
pv_cap_per_sqkm = 81.5                  # MW  (DEA 2030 utility-scale)
pv_percent_land_available = 0.03        # share of suitable area
wind_cap_per_sqkm = 27                  # MW
wind_percent_land_available = 0.03      # share of suitable area
rooftop_cap_per_sqkm = 100              # MW
rooftop_percent_area_available = 0.05   # installable share of built-up area

offshore_cap_per_sqkm = 5               # MW/km2
offshore_percent_available = {"shallow": 0.10, "transitional": 0.10, "deep": 0.05}

## Estimate first (no download)

Print each tile's cutout grid size + the total and the peak single tile (which bounds
peak RAM). Adjust `TILES` / `dx_step` / `dy_step` if a tile is too big before
downloading anything.

In [ ]:
_ = run_batch_by_tile(
    TILES, geo_file, timeframe, filename,
    dx=dx_step, dy=dy_step, bbox_buffer=bbox_buffer,
    estimate_only=True)

## Run the batch

One call does everything: per tile it estimates, builds/reuses the cutout, assigns
coords, runs the capacity factors and (if `run_potentials`) the GIS potentials, then
frees the cutout. After the loop it writes the combined CSVs under the base
`filename` and returns the combined dataframes + per-tile maps.

The first run downloads ERA5 and will take a while - 8 tiles, with `australia` the
largest. Re-running reuses each tile's cutout `.nc` and the ERA5 chunk cache; set
`skip_if_combined_exists=True` to also skip tiles whose combined CSVs already exist.

In [ ]:
result = run_batch_by_tile(
    TILES, geo_file, timeframe, filename,
    dx=dx_step, dy=dy_step, bbox_buffer=bbox_buffer,
    offshore_file=offshore_file, bathymetry_file=bathymetry_file, admin=admin,
    solar_panel=solar_panel, onshore_turbine=onshore_turbine, offshore_turbine=offshore_turbine,
    pv_slope=pv_slope, pv_azimuth=pv_azimuth, optimal_tilt=pv_optimal_tilt,
    rooftop_segments=rooftop_segments, rooftop_shares=rooftop_shares,
    excluder=excluder, cities=cities, tile_layers=TILE_LAYERS,
    pv_cap_per_sqkm=pv_cap_per_sqkm, pv_percent_land_available=pv_percent_land_available,
    wind_cap_per_sqkm=wind_cap_per_sqkm, wind_percent_land_available=wind_percent_land_available,
    rooftop_cap_per_sqkm=rooftop_cap_per_sqkm, rooftop_percent_area_available=rooftop_percent_area_available,
    offshore_cap_per_sqkm=offshore_cap_per_sqkm, offshore_percent_available=offshore_percent_available,
    usable_threshold=usable_threshold, usable_threshold_rooftop=usable_threshold_rooftop,
    run_pv=True, run_wind_onshore=True, run_wind_offshore=True,
    run_temperature=True, run_potentials=run_potentials,
    skip_if_combined_exists=False,
    output_dir=output_dir, color=map_color)

print("tiles run    :", result["tiles_run"])
print("tiles skipped:", result["tiles_skipped"])
print("missing codes:", result["missing_codes"])
print("combined CF families:", sorted(result["cf"].keys()))

## Combined outputs

Combined CSVs are on disk under the base `filename` (e.g.
`{timeframe}_{filename}_pv_opt.csv`, `{filename}_potentials_combined.csv`). Below: a
quick sanity check that a combined CF frame carries every region's column, and the
combined availability map across all tiles.

In [ ]:
# Sanity check: the combined PV-opt frame should have one column per region across ALL tiles.
if "pv_opt" in result["cf"]:
    pv_opt = result["cf"]["pv_opt"]
    all_codes = sorted({c for codes in TILES.values() for c in codes})
    print(f"pv_opt: {pv_opt.shape[1]} region columns, {pv_opt.shape[0]} timesteps")
    print("missing region columns:", sorted(set(all_codes) - set(pv_opt.columns)))

if result["potentials_combined"] is not None:
    display(result["potentials_combined"])

In [ ]:
# Combined availability map across all tiles (only if GIS potentials ran).
if result["maps"]:
    combine_and_plot_tile_maps(result["maps"], which="land", geo_file=geo_file, color=map_color)
else:
    print("No maps (run_potentials was False).")

## Scale up to a full year

Once the dry run looks right, set `timeframe = "2018"` in the config cell and re-run.
The dry-run day's cutouts and the ERA5 chunk cache are reused; the full year downloads
the remaining hours into the same cache. Everything else is unchanged.

## Transfer archive for GENeSYS-MOD

Bundles what the model needs from this run into `output/zip_transfer/{timeframe}_{filename}_genesys_inputs.zip`: temperature, heating and cooling series, heat-pump COPs, usable-site profiles (PV incl. rooftop segments, onshore and offshore wind) and the usable-area potentials, plus a `manifest.json` describing every file. Drop it into the data repo's input folder.

In [ ]:
zip_path = create_transfer_zip(filename, timeframe, rooftop_segments=rooftop_segments)